<a href="https://colab.research.google.com/github/jdasam/ant5015/blob/2026/notebooks/5th_week_autotagging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Music Auto-Tagging with CNN

In this notebook, we build a music auto-tagging system using a CNN on mel spectrograms.

**Dataset:** MagnaTagATune (MTAT) — 8,000 tracks, 50 binary tags (genre, mood, instrumentation)

**Pipeline:**
1. Load and explore the dataset
2. Implement `MTATDataset` (PyTorch `Dataset`)
3. Implement a CNN-based `AutoTagger` model
4. Train with Binary Cross-Entropy loss
5. Evaluate: accuracy, precision, recall, ROC curve

In [1]:
import torch
import torchaudio
import torch.nn as nn
import IPython.display as ipd
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
!pip install --upgrade gdown
!gdown 15e9E3oZdudErkPKwb0rCAiZXkPxdZkV6
!unzip -q mtat_8000.zip

Downloading...
From (original): https://drive.google.com/uc?id=15e9E3oZdudErkPKwb0rCAiZXkPxdZkV6
From (redirected): https://drive.google.com/uc?id=15e9E3oZdudErkPKwb0rCAiZXkPxdZkV6&confirm=t&uuid=eaa1cdcd-6a66-46f4-93a3-17c76adc78ad
To: /content/mtat_8000.zip
100% 921M/921M [00:06<00:00, 144MB/s]


## 1. Dataset Exploration

In [3]:
data_dir = Path('MTAT_SMALL')
assert data_dir.exists()

mp3_fns = list(data_dir.rglob('*.mp3'))
print(f"Total tracks: {len(mp3_fns)}")

mp3_fn = mp3_fns[0]
y, sr = torchaudio.load(mp3_fn)
print(mp3_fn, sr)
ipd.Audio(y, rate=sr)

Total tracks: 8000
MTAT_SMALL/e/touchinggrace-submission-09-last_nights_dream__the_remembrance-262-291.mp3 16000


In [4]:
# Inspect labels (multi-hot tag annotation)
df = pd.read_csv('MTAT_SMALL/meta.csv', index_col=[0])
print(f"Tag vocabulary ({len(df.columns) - 2} tags):")
print(df.columns.values[1:-1])
df.head()

Tag vocabulary (50 tags):
['singer' 'harpsichord' 'sitar' 'heavy' 'foreign' 'no piano' 'classical'
 'female' 'jazz' 'guitar' 'quiet' 'solo' 'folk' 'ambient' 'new age'
 'synth' 'drum' 'bass' 'loud' 'string' 'opera' 'fast' 'country' 'violin'
 'electro' 'trance' 'chant' 'strange' 'modern' 'hard' 'harp' 'pop'
 'female vocal' 'piano' 'orchestra' 'eastern' 'slow' 'male' 'vocal'
 'no singer' 'india' 'rock' 'dance' 'cello' 'techno' 'flute' 'beat' 'soft'
 'choir' 'baroque']


,clip_id,singer,harpsichord,sitar,heavy,foreign,no piano,classical,female,jazz,...,rock,dance,cello,techno,flute,beat,soft,choir,baroque,mp3_path
20552,45147,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,2/zephyrus-angelus-11-ave_maria__virgo_serena_...
3899,8539,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,a/tilopa-pictures_of_silence-02-ni-175-204.mp3
8996,19647,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5/arthur_yoria-of_the_lovely-04-several_mistak...
4055,8856,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,8/stargarden-music_for_modern_listening-02-per...
6361,13834,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,a/dac_crowell-the_mechanism_of_starlight-03-me...


In [10]:
df[df['mp3_path'] == str(mp3_fn.relative_to('MTAT_SMALL/'))]

,clip_id,singer,harpsichord,sitar,heavy,foreign,no piano,classical,female,jazz,...,rock,dance,cello,techno,flute,beat,soft,choir,baroque,mp3_path
18147,39795,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,e/touchinggrace-submission-09-last_nights_drea...


In [12]:
# Check the tags for a specific track
df[df['mp3_path'] == str(mp3_fn.relative_to('MTAT_SMALL/'))].values

array([[39795, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0,
        'e/touchinggrace-submission-09-last_nights_dream__the_remembrance-262-291.mp3']],
      dtype=object)

## 2. MTATDataset

Implement a PyTorch `Dataset` that:
- Splits data by sub-directory ID (train / valid / test)
- Loads all audio files into memory upfront
- Returns `(waveform_tensor, label_tensor)` pairs

| Split | Sub-directory IDs |
|-------|-------------------|
| train | 0–9, a–c |
| valid | d |
| test  | e, f, g |

In [44]:
class MTATDataset:
    def __init__(self, dir_path, split='train', num_max_data=6000, sr=16000):
        self.dir = Path(dir_path)
        self.labels = pd.read_csv(self.dir / "meta.csv", index_col=[0])
        self.sr = sr

        if split == "train":
            sub_dir_ids = ['0','1','2','3','4','5','6','7','8','9','a','b','c']
        elif split == 'valid':
            sub_dir_ids = ['d']
        elif split == 'test':
            sub_dir_ids = ['e','f','g']
        else:
            raise NotImplementedError

        is_in_set = [x[0] in sub_dir_ids for x in self.labels['mp3_path'].values.astype(str)]
        self.labels = self.labels.iloc[is_in_set][:num_max_data]
        self.vocab = self.labels.columns.values[1:-1]
        self.label_tensor = self.convert_label_to_tensor()
        self.audios = self.load_audio()

    def convert_label_to_tensor(self):
        # TODO: Convert the tag columns (index 1:-1) of self.labels to bool,
        #       then return as a tensor with dtype=torch.float
        # self.labels.
        return torch.tensor(self.labels[self.vocab].values, dtype=torch.float)

    def load_audio(self):
        # TODO: Use tqdm to load all mp3 files with torchaudio.load,
        #       and return a list of audio tensors (first channel only)
        pass

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # TODO: Return the idx-th (audio, label) pair from self.audios and self.label_tensor
        mp3_fn = self.labels.iloc[idx]['mp3_path']
        mp3_fn = self.dir / mp3_fn

        y, sr = torchaudio.load(mp3_fn)
        label = self.label_tensor[idx]
        return y, label


train_set = MTATDataset('MTAT_SMALL')
train_set.vocab
audio, label = train_set[10]
ipd.display(ipd.Audio(audio, rate=train_set.sr, normalize=False))
print("Label:", label)

activated_tag_idxs = torch.where(label)[0]
print("Tags:", train_set.vocab[activated_tag_idxs])

Label: tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Tags: ['classical' 'quiet' 'ambient' 'string' 'harp' 'slow']


In [63]:
dummy = torch.arange(9).reshape(1, 1, -1).to(torch.float)
dummy
cnn = torch.nn.Conv1d(in_channels=1,
                      out_channels=2,
                      kernel_size=3,
                      padding=1)


# cnn(dummy)
cnn.weight.data = torch.ones_like(cnn.weight)
cnn.weight.data[1] = torch.tensor([-1, 0, 1]).reshape(cnn.weight.data[1].shape)
cnn.bias.data = torch.zeros_like(cnn.bias)
out = cnn(dummy)

print(f"Output: \n {out}")
print(f"Output.shape: {out.shape}")

Output: 
 tensor([[[ 1.,  3.,  6.,  9., 12., 15., 18., 21., 15.],
         [ 1.,  2.,  2.,  2.,  2.,  2.,  2.,  2., -7.]]],
       grad_fn=<ConvolutionBackward0>)
Output.shape: torch.Size([1, 2, 9])


In [61]:
cnn.weight.data.shape

torch.Size([2, 1, 3])

## 3. CNN-based AutoTagger

We treat the mel spectrogram as a 2D image and apply a standard CNN classifier.

**`__init__()` — define the following layers:**
- `self.mel` — `MelSpectrogram(n_fft=2048, hop_length=1024, n_mels=80)`: waveform → 2D frequency–time representation
- `self.db` — `AmplitudeToDB()`: convert linear amplitude to dB scale
- `self.conv_stack` — three `Conv2d → ReLU → MaxPool2d(2)` blocks (channels: 1 → 16 → 32 → 64)
- `self.final_pool` — `AdaptiveMaxPool1d(1)`: collapse variable-length time axis to a fixed-size vector
- `self.proj` — `nn.Linear(512, out_size)`: tag classification head

**`forward()` — processing pipeline:**
mel → dB → conv_stack → flatten(C, H axes) → temporal pooling → linear → sigmoid

> **Why sigmoid?** Each tag is an independent binary prediction — sigmoid treats all 50 tags as separate yes/no questions.

In [64]:
import torchaudio

class AutoTagger(nn.Module):
    def __init__(self, out_size=50):
        super().__init__()
        # TODO: Define the layers below
        self.mel = torchaudio.transforms.MelSpectrogram(n_fft=2048, hop_length=1024, n_mels=80)
        self.db  = torchaudio.transforms.AmplitudeToDB()
        # self.conv_stack : nn.Sequential with three Conv2d→ReLU→MaxPool2d(2) blocks
        #   Block 1: Conv2d(1,  16, kernel_size=3)
        #   Block 2: Conv2d(16, 32, kernel_size=3)
        #   Block 3: Conv2d(32, 64, kernel_size=3)
        # self.final_pool : nn.AdaptiveMaxPool1d(1)
        # self.proj       : nn.Linear(512, out_size)
        pass

    def forward(self, x):
        # TODO: Pass x through the pipeline in this order:
        # 1. Mel spectrogram → AmplitudeToDB, then divide by 80 (normalization)
        # 2. conv_stack
        # 3. Flatten: (N, C, H, W) → (N, C*H, W)
        # 4. final_pool → squeeze the last dimension
        # 5. proj → sigmoid, and return
        # pass
        x = self.db(self.mel(x)) / 100
        return x



model = AutoTagger()
out = model(audio)
print("Output shape:", out.shape)  # expected: (50,)

Output shape: torch.Size([1, 80, 456])


## 4. Loss Function: Binary Cross-Entropy

Since each tag is an independent binary label, we use **Binary Cross-Entropy (BCE)**:

$$\text{BCE} = -\frac{1}{N}\sum_{i} \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

We add a small epsilon (`1e-8`) inside the log to prevent numerical instability.

In [ ]:
def get_binary_cross_entropy(pred, target, reduce=True):
    # TODO: Implement the BCE formula:
    # If reduce=True, return loss.mean(); otherwise return the element-wise loss tensor
    pass


# Quick test with a single sample
prob = model(audio)
loss = get_binary_cross_entropy(prob.cpu(), label)
print("BCE loss:", loss)

In [ ]:
# Verify batch shape
train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True)

batch_audio, batch_label = next(iter(train_loader))
out = model(batch_audio)
print("Batch output shape:", out.shape)   # expected: (32, 50)
print("Batch label shape:", batch_label.shape)

## 5. Training

Implement the standard PyTorch training loop:

```
for each epoch:
    for each batch:
        forward pass  →  compute loss  →  backward  →  optimizer.step()  →  zero_grad()
```

In [ ]:
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

train_loader = torch.utils.data.DataLoader(train_set, batch_size=32, shuffle=True)
model = AutoTagger().to(DEV)
optimizer = torch.optim.Adam(model.parameters())

loss_record = []
n_epoch = 20

for epoch in tqdm(range(n_epoch)):
    for batch in tqdm(train_loader, leave=False):
        # TODO: Unpack audio and label from batch, move to DEV
        #       forward → compute loss → backward → optimizer.step() → zero_grad()
        pass

In [ ]:
plt.plot(loss_record)
plt.xlabel("Iteration")
plt.ylabel("BCE Loss")
plt.title("Training Loss")
plt.show()

## 6. Evaluation

Load the validation set and compute predictions.

In [ ]:
valid_set = MTATDataset('MTAT_SMALL', split='valid', num_max_data=1000)
valid_loader = torch.utils.data.DataLoader(valid_set, batch_size=50, shuffle=False)

model = model.cpu()
torch.set_printoptions(sci_mode=False)

batch_audio, label = next(iter(valid_loader))
prob = model(batch_audio)
print("prob shape:", prob.shape)

### 6.1 Accuracy

Since this is a multi-label problem with 50 tags, accuracy is computed **element-wise** across all (sample, tag) pairs.

> Note: always compare against a **null baseline** (predict all-zero) — the dataset is heavily imbalanced.

In [ ]:
threshold = 0.5

# TODO: Binarize prob using threshold to get thresholded_pred,
#       then compute element-wise accuracy against label
#       Hint: mean of (thresholded_pred == label)

accuracy = None
print("Model accuracy:", accuracy)

In [ ]:
# Null baseline: predicting all zeros
null_pred = torch.zeros_like(prob)
null_accuracy = (null_pred == label).float().mean()
print("Null baseline accuracy:", null_accuracy)
print("Positive label ratio:", label.mean().item())

### 6.2 Precision & Recall

Accuracy is misleading for imbalanced labels. We need **precision** and **recall**:

| Metric | Formula | Meaning |
|--------|---------|---------|
| Precision | TP / (TP + FP) | Of predicted positives, how many are correct? |
| Recall    | TP / (TP + FN) | Of actual positives, how many did we catch? |

In [ ]:
threshold = 0.7
thresholded_pred = (prob > threshold).float()

# TODO: Compute precision
# true_positive    = number of cases where both pred and label are 1
# total_pos_pred   = number of cases where pred is 1
# precision        = true_positive / total_pos_pred

precision = None
print("Precision:", precision)

In [ ]:
# TODO: Compute recall
# true_positive  = number of cases where both pred and label are 1
# num_pos_sample = number of cases where label is 1
# recall         = true_positive / num_pos_sample

recall = None
print("Recall:", recall)

### 6.3 ROC Curve

The ROC curve plots **TPR (True Positive Rate)** vs **FPR (False Positive Rate)** across all thresholds.

- **TPR** = TP / (TP + FN) = recall  
- **FPR** = FP / (FP + TN)  
- A random classifier follows the diagonal; a perfect classifier reaches the top-left corner.

In [ ]:
def get_tpr_fpr(pred, target, threshold):
    # TODO: Binarize pred using threshold, then compute TP, TN, FP, FN
    #       Return (tpr, fpr) as floats
    #       tpr = TP / (TP + FN)
    #       fpr = FP / (FP + TN)
    pass


# Quick test
print(get_tpr_fpr(prob, label, 0.5))

In [ ]:
# Plot the ROC curve
tprs, fprs = [], []

for th in reversed(torch.linspace(0, 1, 500)):
    tpr, fpr = get_tpr_fpr(prob, label, th)
    tprs.append(tpr)
    fprs.append(fpr)

plt.figure(figsize=(6, 6))
plt.plot(fprs, tprs)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()
plt.show()

## 7. Inference

Run the trained model on individual tracks and listen to the predictions.

In [ ]:
# Predict tags for a single track
audio_sample, true_label = valid_set[70]
threshold = 0.25

pred = model(audio_sample)
pred_tag_ids = torch.where(pred > threshold)[0]

print("Predicted tags:", valid_set.vocab[pred_tag_ids])
print("True tags:     ", valid_set.vocab[torch.where(true_label)[0]])

ipd.Audio(audio_sample * 0.1, rate=valid_set.sr, normalize=False)

In [ ]:
# Find the hardest samples (highest loss) in the validation set
loss_per_sample = []
for batch_audio, batch_label in valid_loader:
    prob_batch = model(batch_audio)
    loss = get_binary_cross_entropy(prob_batch.cpu(), batch_label, reduce=False)
    loss_per_sample.append(loss.mean(dim=-1))

loss_per_sample = torch.cat(loss_per_sample)
hardest_idx = loss_per_sample.argmax().item()
print(f"Hardest sample index: {hardest_idx}, loss: {loss_per_sample[hardest_idx]:.4f}")